In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import numpy as np
import copy


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASELINE_CKPT = "/kaggle/input/baseline-75-dbc/densenet_bc100_baseline (1).pt"
PRUNED_CKPT   = "pruned_DenseNetBC100_Taylor.pt"

BATCH_SIZE = 128
PRUNE_RATIO = 0.3          # 30% channel pruning
FINE_TUNE_EPOCHS = 60
LR = 0.01


In [3]:
class Bottleneck(nn.Module):
    def __init__(self, in_planes, growth_rate):
        super().__init__()
        inter_planes = 4 * growth_rate
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, inter_planes, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(inter_planes)
        self.conv2 = nn.Conv2d(inter_planes, growth_rate, 3, padding=1, bias=False)

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.conv2(F.relu(self.bn2(out)))
        return torch.cat([x, out], 1)


class Transition(nn.Module):
    def __init__(self, in_planes, out_planes):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_planes)
        self.conv = nn.Conv2d(in_planes, out_planes, 1, bias=False)

    def forward(self, x):
        out = self.conv(F.relu(self.bn(x)))
        return F.avg_pool2d(out, 2)


class DenseNetBC(nn.Module):
    def __init__(self, depth=100, growth_rate=12, reduction=0.5, num_classes=100):
        super().__init__()
        n = (depth - 4) // 6
        num_planes = 2 * growth_rate

        self.conv1 = nn.Conv2d(3, num_planes, 3, padding=1, bias=False)

        self.dense1 = self._make_dense(num_planes, n, growth_rate)
        num_planes += n * growth_rate
        self.trans1 = Transition(num_planes, int(num_planes * reduction))
        num_planes = int(num_planes * reduction)

        self.dense2 = self._make_dense(num_planes, n, growth_rate)
        num_planes += n * growth_rate
        self.trans2 = Transition(num_planes, int(num_planes * reduction))
        num_planes = int(num_planes * reduction)

        self.dense3 = self._make_dense(num_planes, n, growth_rate)
        num_planes += n * growth_rate

        self.bn = nn.BatchNorm2d(num_planes)
        self.fc = nn.Linear(num_planes, num_classes)

    def _make_dense(self, in_planes, n, growth_rate):
        layers = []
        for _ in range(n):
            layers.append(Bottleneck(in_planes, growth_rate))
            in_planes += growth_rate
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.trans1(self.dense1(x))
        x = self.trans2(self.dense2(x))
        x = self.dense3(x)
        x = F.relu(self.bn(x))
        x = F.adaptive_avg_pool2d(x, 1)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [4]:
MEAN = [0.5071, 0.4867, 0.4408]
STD  = [0.2675, 0.2565, 0.2761]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

train_set = torchvision.datasets.CIFAR100("./data", train=True, download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR100("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)


100%|██████████| 169M/169M [00:01<00:00, 85.6MB/s]


In [5]:
model = DenseNetBC().to(DEVICE)
model.load_state_dict(torch.load(BASELINE_CKPT, map_location="cpu", weights_only=True))
model.eval()

print("✅ Baseline model loaded")


✅ Baseline model loaded


In [6]:
def compute_taylor_scores(model, loader, criterion):
    scores = {}
    model.train()

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        model.zero_grad()

        loss = criterion(model(images), labels)
        loss.backward()

        for name, m in model.named_modules():
            if isinstance(m, nn.Conv2d):
                if m.weight.grad is not None:
                    score = (m.weight * m.weight.grad).abs().sum(dim=(1,2,3))
                    scores.setdefault(m, 0)
                    scores[m] += score.detach().cpu()
        break  # single batch is enough (standard Taylor pruning)

    return scores


In [7]:
def prune_model(model, scores, prune_ratio):
    pruned = copy.deepcopy(model)

    for m, score in scores.items():
        num_prune = int(len(score) * prune_ratio)
        if num_prune == 0:
            continue

        threshold = torch.topk(score, num_prune, largest=False).values.max()

        mask = score > threshold
        m.weight.data = m.weight.data[mask]

    return pruned


In [8]:
criterion = nn.CrossEntropyLoss()

print("🔍 Computing Taylor importance...")
scores = compute_taylor_scores(model, train_loader, criterion)

print("✂️ Pruning model...")
pruned_model = prune_model(model, scores, PRUNE_RATIO)
pruned_model = pruned_model.to(DEVICE)


🔍 Computing Taylor importance...
✂️ Pruning model...


In [9]:
from tqdm import tqdm

optimizer = torch.optim.SGD(
    pruned_model.parameters(),
    lr=LR,
    momentum=0.9,
    weight_decay=1e-4
)

best_acc = 0.0

for epoch in range(60):

    # ===================== TRAIN =====================
    pruned_model.train()
    running_loss = 0.0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch [{epoch+1}/{FINE_TUNE_EPOCHS}] TRAIN",
        leave=False
    )

    for images, labels in train_bar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = pruned_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        train_bar.set_postfix(
            loss=f"{running_loss / (train_bar.n + 1):.4f}"
        )

    avg_train_loss = running_loss / len(train_loader)

    # ===================== EVAL =====================
    pruned_model.eval()
    correct, total = 0, 0

    eval_bar = tqdm(
        test_loader,
        desc=f"Epoch [{epoch+1}/{FINE_TUNE_EPOCHS}] EVAL",
        leave=False
    )

    with torch.no_grad():
        for images, labels in eval_bar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = pruned_model(images)
            preds = outputs.argmax(1)

            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            eval_bar.set_postfix(
                acc=f"{100. * correct / total:.2f}%"
            )

    acc = 100. * correct / total

    # ===================== SAVE BEST =====================
    if acc > best_acc:
        best_acc = acc
        torch.save(pruned_model.state_dict(), PRUNED_CKPT)

    print(
        f"Epoch {epoch+1}/{FINE_TUNE_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Acc: {acc:.2f}% | "
        f"Best: {best_acc:.2f}%"
    )


Epoch 1/60 | Train Loss: 0.0193 | Val Acc: 75.77% | Best: 75.77%


Epoch 2/60 | Train Loss: 0.0094 | Val Acc: 75.97% | Best: 75.97%


Epoch 3/60 | Train Loss: 0.0078 | Val Acc: 75.92% | Best: 75.97%


Epoch 4/60 | Train Loss: 0.0076 | Val Acc: 75.73% | Best: 75.97%


Epoch 5/60 | Train Loss: 0.0072 | Val Acc: 75.86% | Best: 75.97%


...

Epoch 56/60 | Train Loss: 0.0060 | Val Acc: 75.24% | Best: 75.97%


Epoch 57/60 | Train Loss: 0.0059 | Val Acc: 75.25% | Best: 75.97%


Epoch 58/60 | Train Loss: 0.0059 | Val Acc: 75.37% | Best: 75.97%


Epoch 59/60 | Train Loss: 0.0059 | Val Acc: 75.20% | Best: 75.97%


Epoch 60/60 | Train Loss: 0.0059 | Val Acc: 75.10% | Best: 75.97%


In [10]:
print("\n========== PRUNING SUMMARY ==========")
print(f"Pruning Ratio     : {PRUNE_RATIO}")
print(f"Best Accuracy     : {best_acc:.2f}%")
print(f"Saved Model       : {PRUNED_CKPT}")
print("====================================")



========== PRUNING SUMMARY ==========
Pruning Ratio     : 0.3
Best Accuracy     : 75.97%
Saved Model       : pruned_DenseNetBC100_Taylor.pt
